# SageMaker Calibration Jobs

Lanza jobs de calibracion post-entrenamiento usando `src/calibration_entrypoint.py`.

In [ ]:
from datetime import datetime, timezone
from sagemaker.core.helper.session_helper import Session, s3_path_join
import boto3

sagemaker_session = Session(default_bucket_prefix='congestion-bogota-training')
region = 'us-east-1'
role = 'arn:aws:iam::195419001736:role/SagemakerExecutionRole'
bucket_name = 'sagemaker-congestion-bogota'
sm = boto3.client('sagemaker', region_name=region)
s3 = boto3.client('s3', region_name=region)

efs_id = 'fs-099e29197c0a3e353'
efs_path = '/data/datos_procesados'
vpc_config = {
    'subnets': ['subnet-017c79447d5151efe'],
    'security_group_ids': ['sg-0962698c0c8352ddd', 'sg-03590ecb70a82d570'],
}

print(region)
print(role)


In [ ]:
import os
import tarfile
import tempfile

TAR_SOURCE_FILENAME = 'sourcedir.tar.gz'

def tar_and_upload_source_code(source_path: str, bucket: str, s3_key_prefix: str='code'):
    source_path = os.path.abspath(source_path)
    include_files = ['requirements-sagemaker.txt']
    with tempfile.TemporaryDirectory() as tmpdir:
        tar_path = os.path.join(tmpdir, TAR_SOURCE_FILENAME)
        with tarfile.open(tar_path, 'w:gz', dereference=True) as tar:
            src_root = os.path.join(source_path, 'src')
            for root, dirs, files in os.walk(src_root):
                dirs[:] = [d for d in dirs if d != '__pycache__']
                for fname in files:
                    if fname.endswith('.py'):
                        abs_path = os.path.join(root, fname)
                        rel_path = os.path.relpath(abs_path, source_path)
                        tar.add(abs_path, arcname=rel_path)
            for rel_path in include_files:
                abs_path = os.path.join(source_path, rel_path)
                if os.path.exists(abs_path):
                    tar.add(abs_path, arcname=rel_path)
        s3_key = f'{s3_key_prefix}/{TAR_SOURCE_FILENAME}'
        s3.upload_file(tar_path, bucket, s3_key)
        return s3_path_join('s3://', bucket, s3_key)

source_code_uri = tar_and_upload_source_code('..', bucket_name, 'congestion-bogota/calibration/code')
print(source_code_uri)


In [ ]:
image = '195419001736.dkr.ecr.us-east-1.amazonaws.com/congestion-bogota-sagemaker:temporal-features-20260331-194332'
project_s3_path = s3_path_join('s3://', bucket_name, 'congestion-bogota')

calibration_jobs = [
    {
        'model_type': 'GConvLSTM',
        'model_artifact_path': 's3://ine-projects-incoming-data/congestion-prediction/co/movilidadbogota/artifacts/model_gconvlstm_2026_04_10.tar.gz',
        'instance_type': 'ml.g5.4xlarge',
    },
    {
        'model_type': 'GConvGRU',
        'model_artifact_path': 's3://ine-projects-incoming-data/congestion-prediction/co/movilidadbogota/artifacts/model_gconvgru_2026_04_10.tar.gz',
        'instance_type': 'ml.g6.4xlarge',
    },
    {
        'model_type': 'TGCN',
        'model_artifact_path': 's3://ine-projects-incoming-data/congestion-prediction/co/movilidadbogota/artifacts/model_tgcn_2026_04_10.tar.gz',
        'instance_type': 'ml.g6e.4xlarge',
    },
]

for item in calibration_jobs:
    item['job_name'] = f"congestion-bogota-calibration-{item['model_type'].lower()}-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
    item['s3_output_path'] = s3_path_join(project_s3_path, 'calibration-runs', item['model_type'].lower())

calibration_jobs


In [ ]:
NUMERIC_REGEX = r'([-+0-9eE\.]+)'
metric_names = [
    'raw_count', 'raw_mae', 'raw_accuracy_exact', 'raw_accuracy_pm1',
    'qmap_global_count', 'qmap_global_mae', 'qmap_global_accuracy_exact', 'qmap_global_accuracy_pm1',
    'hurdle_qmap_count', 'hurdle_qmap_mae', 'hurdle_qmap_accuracy_exact', 'hurdle_qmap_accuracy_pm1',
    'hybrid_0_6_count', 'hybrid_0_6_mae', 'hybrid_0_6_accuracy_exact', 'hybrid_0_6_accuracy_pm1',
    'hybrid_0_8_count', 'hybrid_0_8_mae', 'hybrid_0_8_accuracy_exact', 'hybrid_0_8_accuracy_pm1',
]
for level in range(5):
    metric_names.extend([
        f'hurdle_qmap_level_{level}_count',
        f'hurdle_qmap_level_{level}_mae',
        f'hurdle_qmap_level_{level}_accuracy_exact',
        f'hurdle_qmap_level_{level}_accuracy_pm1',
    ])
assert len(metric_names) <= 40, len(metric_names)
metric_definitions_payload = [
    {'Name': name, 'Regex': rf'{name}={NUMERIC_REGEX}'}
    for name in metric_names
]
len(metric_definitions_payload)


In [ ]:
launched_jobs = []
for item in calibration_jobs:
    hyper_parameters = {
        'sagemaker_program': 'src/calibration_entrypoint.py',
        'sagemaker_submit_directory': source_code_uri,
        'model-artifact-path': item['model_artifact_path'],
        'train-ratio': '0.8',
        'input-seq-len': '150',
        'output-seq-len': '50',
        'node-batch-size': '500',
        'include-weather': 'True',
        'include-velocity-target': 'True',
        'use-minimal-datasets': 'True',
        'batch-size': '16',
        'num-workers': '4',
        'pin-memory': 'True',
        'prefetch-factor': '2',
        'persistent-workers': 'True',
        'drop-last': 'False',
        'cache-local': 'True',
        'cache-dir': '/opt/ml/input/data/cache',
        'calibration-max-pairs': '200000',
        'qmap-quantiles': '201',
        'calibration-thresholds': '0.6,0.8',
        'selection-metric': 'accuracy_exact',
    }
    request = {
        'TrainingJobName': item['job_name'],
        'RoleArn': role,
        'AlgorithmSpecification': {
            'TrainingImage': image,
            'TrainingInputMode': 'File',
            'MetricDefinitions': metric_definitions_payload,
            'EnableSageMakerMetricsTimeSeries': True,
        },
        'HyperParameters': hyper_parameters,
        'InputDataConfig': [
            {
                'ChannelName': 'train',
                'DataSource': {
                    'FileSystemDataSource': {
                        'FileSystemId': efs_id,
                        'FileSystemType': 'EFS',
                        'FileSystemAccessMode': 'ro',
                        'DirectoryPath': efs_path,
                    }
                },
            },
        ],
        'OutputDataConfig': {
            'S3OutputPath': item['s3_output_path'],
        },
        'ResourceConfig': {
            'InstanceType': item['instance_type'],
            'InstanceCount': 1,
            'VolumeSizeInGB': 100,
        },
        'StoppingCondition': {
            'MaxRuntimeInSeconds': 6 * 60 * 60,
        },
        'VpcConfig': {
            'Subnets': vpc_config['subnets'],
            'SecurityGroupIds': vpc_config['security_group_ids'],
        },
    }
    resp = sm.create_training_job(**request)
    launched_jobs.append({
        'model_type': item['model_type'],
        'job_name': item['job_name'],
        'instance_type': item['instance_type'],
        'model_artifact_path': item['model_artifact_path'],
        'training_job_arn': resp['TrainingJobArn'],
    })
    print(item['job_name'], item['instance_type'])

launched_jobs


In [ ]:
for item in launched_jobs:
    desc = sm.describe_training_job(TrainingJobName=item['job_name'])
    print(item['job_name'])
    print(desc['TrainingJobStatus'], desc['SecondaryStatus'])
    print(desc.get('FailureReason'))
    print()
